### Loading Previously Cleaned Zillow Dataset 

In [1]:
import pandas as pd

# loading cleaned zillow dataset 
zillow_df = pd.read_csv('zillow_cleaned.csv')
print(zillow_df.head())


   Unnamed: 0       zpid                                       address  \
0           0   17264897             979 Kevin Ave, Redlands, CA 92373   
1           1   20021372      13114 Addison St, Sherman Oaks, CA 91423   
2           2   20009320  6032 Goodland Ave, North Hollywood, CA 91606   
3           3  460204882          3325 Tonopah St, Oceanside, CA 92054   
4           4   20769150     963 Pine Grove Ave, Los Angeles, CA 90042   

     price  beds  baths  area_sqft   latitude   longitude          status  \
0   447000   3.0    2.0     1300.0  34.040520 -117.186195  House for sale   
1  2795000   4.0    5.0     2900.0  34.160885 -118.418770  House for sale   
2  1718000   4.0    3.0     3025.0  34.180450 -118.411280  House for sale   
3   899999   3.0    3.0     1682.0  33.214260 -117.341500     Coming soon   
4   995000   2.0    2.0     1271.0  34.132890 -118.186220  House for sale   

   ...  zestimate                                         detail_url  \
0  ...   455500.0  h

### Loading State Income Tax Rates and Property Taxes by State and County Datasets 

In [2]:
# loading state income tax rates dataset
state_inc_df = pd.read_csv('2026 State Income Tax Rates and Brackets  Tax Foundation (1).csv')
# printing first few rows of data
print(state_inc_df.head())

# loading property taxes dataset 
prop_tax_df = pd.read_csv('Property Taxes by State and County, 2026  Tax Foundation Maps.csv')
# printing first few rows of data 
print(prop_tax_df.head())

                   State Single Filer (Rates) Unnamed: 2  \
0  Alabama (a, b, c, uu)                2.00%          >   
1              - Alabama                4.00%          >   
2              - Alabama                5.00%          >   
3                 Alaska                 none        NaN   
4  Arizona (e, f, u, vv)                2.50%          >   

  Single Filer (Brackets) Married Filing Jointly (Rates) Unnamed: 5  \
0                      $0                          2.00%          >   
1                    $500                          4.00%          >   
2                  $3,000                          5.00%          >   
3                     NaN                           none        NaN   
4                      $0                          2.50%          >   

  Married Filing Jointly (Brackets) Standard Deduction (Single)  \
0                                $0                      $3,000   
1                            $1,000                         NaN   
2          

### Cleaning State Income Tax Rates Dataset

In [3]:
import re
import numpy as np

# columns to drop 
col_drop = [
    'Unnamed: 2',                       
    'Unnamed: 5',                       
    'Standard Deduction (Single)',
    'Standard Deduction (Couple)',
    'Personal Exemption (Single)',
    'Personal Exemption (Couple)',
    'Personal Exemption (Dependent)'
]

# dropping > column and deduction columns 
state_inc_df = state_inc_df.drop(columns = col_drop, errors = 'ignore')

# checking df with dropped columns 
# print(state_inc_df.head())

# dropping $, %, and commas 
state_inc_df = state_inc_df.replace(['\$' , '\%', '\,'], '', regex = True)

# need to fix state names 
state_inc_df['State'] = (
    state_inc_df['State']
    .str.replace(r'\(.*?\)', '', regex = True)
    .str.replace('-', '', regex = False)
    .str.strip()
)

# count missing values for each column 
print("Missing values per column:")
print(state_inc_df.isna().sum())

# drop empty rows, rows with None, and n.a 
state_inc_df = state_inc_df.replace(
    ['None', 'none', 'n.a', 'n.a.', ''], 
    np.nan
)
state_inc_df = state_inc_df.dropna(how = 'any')

# fix column names 
state_inc_df.rename(columns = {'State' : 'state', 'Single Filer (Rates)' : 'single_filer_rates', 
                               'Single Filer (Brackets)' : 'single_filer_brackets', 
                               'Married Filing Jointly (Rates)' : 'married_filing_jointly_rates',
                               'Married Filing Jointly (Brackets)' : 'married_filing_jointly_brackets'}, inplace = True)

# checking clean df 
print(state_inc_df.head())

Missing values per column:
State                                 0
Single Filer (Rates)                  5
Single Filer (Brackets)              14
Married Filing Jointly (Rates)        5
Married Filing Jointly (Brackets)    13
dtype: int64
      state single_filer_rates single_filer_brackets  \
0   Alabama               2.00                     0   
1   Alabama               4.00                   500   
2   Alabama               5.00                  3000   
4   Arizona               2.50                     0   
5  Arkansas               2.00                     0   

  married_filing_jointly_rates married_filing_jointly_brackets  
0                         2.00                               0  
1                         4.00                            1000  
2                         5.00                            6000  
4                         2.50                               0  
5                         2.00                               0  


### Cleaning Property Tax Dataset

In [4]:
# getting rid of $, and commas
prop_tax_df['med_housing_value'] = pd.to_numeric(prop_tax_df['Median Housing Value, 2024 ($)'].str.replace(r'[$,]', '', regex = True), errors = 'coerce')

# getting rid of % 
prop_tax_df['prop_tax_rate'] = pd.to_numeric(prop_tax_df['Effective Property Tax Rate (2024)'].str.replace('%', '', regex = True), errors = 'coerce')

# note: original columns were not dropped yet to ensure correct values 

# printing dataset after cleaning 
# print(prop_tax_df.head())

# changing column name 
prop_tax_df.rename(columns = {'Median Property Taxes Paid, 2024 ($)(5-Year Estimate)' : 'med_prop_tax_paid',
                               'State' : 'state', 'County' : 'county'}, inplace = True)

# dropping duplicate columns 
# columns to drop 
col_drop = [
    'Median Housing Value, 2024 ($)',
    'Effective Property Tax Rate (2024)'
]

# dropping > column and deduction columns 
prop_tax_df = prop_tax_df.drop(columns = col_drop, errors = 'ignore')


# printing dataset after dropping
print(prop_tax_df.head())

     state          county med_prop_tax_paid  med_housing_value  prop_tax_rate
0  Alabama  Autauga County               627           207200.0           0.28
1  Alabama  Baldwin County               960           316900.0           0.29
2  Alabama  Barbour County               462           112900.0           0.31
3  Alabama     Bibb County               298           145700.0           0.20
4  Alabama   Blount County               523           175200.0           0.27


### Exporting Tax CSV 

In [31]:
state_inc_df.to_csv('cleaned_state_income_tax.csv', index = False)
prop_tax_df.to_csv('cleaned_property_tax.csv', index = False)
print("Saved: cleaned_state_income_tax.csv")
print("Saved: cleaned_property_tax.csv")

Saved: cleaned_state_income_tax.csv
Saved: cleaned_property_tax.csv


### Adding Median Household by Income Dataset 

In [13]:
# loading median household income by state dataset
med_household_inc_df = pd.read_csv('median-household-income-by-state-2026.csv')

# printing first few rows of dataset to check 
# print(med_household_inc_df.head())

# dropping state flag code column 
med_household_inc_df = med_household_inc_df.drop(columns = 'stateFlagCode', errors = 'ignore')

# printing first few rows of dataset to check 
# print(med_household_inc_df.head())

# fix column names 
med_household_inc_df.rename(columns = {'MedianHouseholdIncomeInOnePersonHousehold_2023' : 'median_household_income_one_person', 
                                       'MedianHouseholdIncomeInTwoPersonHousehold_2023' : 'median_household_income_two_person',
                                       'MedianHouseholdIncomeInThreePersonHousehold_2023' : 'median_household_income_three_person',
                                       'MedianHouseholdIncomeInFourPersonHousehold_2023' : 'median_household_income_four_person'}, inplace = True)

# printing first few rows of dataset to check 
print(med_household_inc_df.head())

                  state  median_household_income_one_person  \
0  District of Columbia                             75814.0   
1              Maryland                             52519.0   
2            California                             50251.0   
3              Colorado                             49447.0   
4            New Jersey                             48523.0   

   median_household_income_two_person  median_household_income_three_person  \
0                            147683.0                              183198.0   
1                            109285.0                              125724.0   
2                            101959.0                              113943.0   
3                            104126.0                              121291.0   
4                            104651.0                              130685.0   

   median_household_income_four_person  
0                             215972.0  
1                             150702.0  
2                      

### Master Dataset

In [34]:
# converting tax paid column to number 
prop_tax_df['med_prop_tax_paid'] = pd.to_numeric(
    prop_tax_df['med_prop_tax_paid'].astype(str).str.replace(r'[$,]', '', regex = True), 
    errors = 'coerce'
)

# property taxes are averaged per state so they don't duplicate zillow cities
prop_tax_state_avg = prop_tax_df.groupby('state', as_index = False).agg({
    'med_housing_value': 'mean',
    'prop_tax_rate': 'mean',
    'med_prop_tax_paid': 'mean' 
})

# rename the average columns
prop_tax_state_avg.rename(columns = {
    'med_housing_value': 'state_avg_housing_value',
    'prop_tax_rate': 'state_avg_prop_tax_rate'
}, inplace = True)


# zillow + property taxes 
merged_df = pd.merge(zillow_df, prop_tax_state_avg, left_on = 'state_name', right_on = 'state', how = 'left')\

# keep highest tax bracket 
state_inc_unique = state_inc_df.sort_values('single_filer_rates').drop_duplicates(subset = ['state'], keep = 'last')

# result + income taxes 
merged_df = pd.merge(merged_df, state_inc_unique, on = 'state', how = 'left')

# clean up the redundant 'state' column from the merge
if 'state' in merged_df.columns:
    merged_df.drop(columns = ['state'], inplace = True)

# export final dataset
merged_df.to_csv('master_zillow_with_taxes.csv', index = False)
print(f"\nSaved master_zillow_with_taxes.csv\nTotal Rows: {len(merged_df)}")


Saved master_zillow_with_taxes.csv
Total Rows: 2159


### New Master Dataset with Median Income Added 

In [17]:
# reading tax bracket column as a number 
state_inc_df['single_filer_brackets'] = pd.to_numeric(
    state_inc_df['single_filer_brackets'].astype(str).str.replace(r'[$,]', '', regex=True), 
    errors='coerce'
)

# convert all to math numbers 
prop_tax_df['med_prop_tax_paid'] = pd.to_numeric(prop_tax_df['med_prop_tax_paid'].astype(str).str.replace(r'[$,]', '', regex=True), errors='coerce')
prop_tax_df['med_housing_value'] = pd.to_numeric(prop_tax_df['med_housing_value'].astype(str).str.replace(r'[$,]', '', regex=True), errors='coerce')
prop_tax_df['prop_tax_rate'] = pd.to_numeric(prop_tax_df['prop_tax_rate'].astype(str).str.replace(r'[%,]', '', regex=True), errors='coerce')

# group by state and find the median to get rid of outliers
prop_tax_state_avg = prop_tax_df.groupby('state', as_index=False).agg({
    'med_housing_value': 'median',
    'prop_tax_rate': 'median',
    'med_prop_tax_paid': 'median' 
})

# average median income for 1 to 4 person households
income_cols = [
    'median_household_income_one_person', 
    'median_household_income_two_person',
    'median_household_income_three_person', 
    'median_household_income_four_person'
]
med_household_inc_df['state_avg_median_income'] = med_household_inc_df[income_cols].mean(axis=1)

# finding realistic tax bracket 
# merge tax dataset with our new average income data
tax_with_income = pd.merge(state_inc_df, med_household_inc_df[['state', 'state_avg_median_income']], on='state', how='left')

# keep only brackets where the state average income is >= the bracket start
valid_brackets = tax_with_income[tax_with_income['state_avg_median_income'] >= tax_with_income['single_filer_brackets']]

# keep the highest valid bracket remaining
state_inc_realistic = valid_brackets.sort_values('single_filer_brackets').drop_duplicates(subset=['state'], keep='last')

# rename the columns so they make sense in the master dataset
prop_tax_state_avg.rename(columns={
    'med_housing_value': 'state_median_housing_value',
    'prop_tax_rate': 'state_median_prop_tax_rate'
}, inplace=True)

# merge
final_merged_df = pd.merge(zillow_df, prop_tax_state_avg, left_on='state_name', right_on='state', how='left')

# add new realistic income taxes
final_merged_df = pd.merge(final_merged_df, state_inc_realistic, on='state', how='left')

# clean duplicate columns 
if 'state' in final_merged_df.columns:
    final_merged_df.drop(columns=['state'], inplace=True)

# export new dataset
final_merged_df.to_csv('master_dataset.csv', index=False)

print(f"\nSaved realistic master_dataset.csv")
print(f"Total Rows: {len(final_merged_df)}\n")

# final result check 
print(final_merged_df[['state_name', 'price', 'state_avg_median_income', 'single_filer_rates']].head())


Saved realistic master_dataset.csv
Total Rows: 2159

   state_name    price  state_avg_median_income single_filer_rates
0  California   447000                  98741.5               9.30
1  California  2795000                  98741.5               9.30
2  California  1718000                  98741.5               9.30
3  California   899999                  98741.5               9.30
4  California   995000                  98741.5               9.30
